# Kizzasi Anomaly Detection

One of Kizzasi's primary use cases is **real-time anomaly detection on continuous sensor signals**:

1. **Learn normal behaviour** — train a Mamba2 predictor on a long run of "healthy" signal
2. **Inject anomalies** — introduce spikes, drift, and missing-data bursts
3. **Detect via prediction error** — flag samples where `|predicted - actual|` exceeds a learned threshold
4. **Enforce constraint guardrails** — apply hard physical bounds that immediately reject impossible values
5. **Visualise** — overlay detections on the raw signal

The `kizzasi.ConstraintSpec` API provides neuro-symbolic guardrails that work alongside the learned error threshold, enabling two complementary detection modes:

| Mode | Mechanism | Latency |
|------|-----------|---------|
| Statistical | prediction error > threshold | ~1–5 samples |
| Constraint | value violates hard bounds | 0 samples (immediate) |

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import kizzasi
    KIZZASI_AVAILABLE = True
    print(f'kizzasi {kizzasi.__version__} loaded')
except ImportError:
    KIZZASI_AVAILABLE = False
    print('kizzasi not installed — numpy-stub mode active')

rng = np.random.default_rng(2024)

## 1. Generate Synthetic Sensor Data

We simulate a 3-axis vibration sensor on industrial machinery:
- **Normal**: band-limited noise around 50 Hz and 100 Hz harmonics
- **Anomalies**: spike bursts (bearing fault), slow drift (thermal), and clamp-outs (sensor failure)

In [ ]:
SAMPLE_RATE  = 1000   # Hz (1 kHz vibration sensor)
TRAIN_SEC    = 5.0    # seconds of normal data for warm-up
TEST_SEC     = 2.0    # seconds of mixed normal+anomalous data
N_TRAIN      = int(SAMPLE_RATE * TRAIN_SEC)
N_TEST       = int(SAMPLE_RATE * TEST_SEC)
N_TOTAL      = N_TRAIN + N_TEST

t = np.linspace(0, N_TOTAL / SAMPLE_RATE, N_TOTAL, endpoint=False)

# --- Normal signal (3-sensor, but we demo on sensor 0) ---
normal = (
    0.6 * np.sin(2 * np.pi * 50  * t) +
    0.3 * np.sin(2 * np.pi * 100 * t) +
    0.15 * np.sin(2 * np.pi * 150 * t) +
    0.05 * rng.standard_normal(N_TOTAL)
).astype(np.float32)

signal = normal.copy()

# Ground-truth anomaly mask (True = anomalous)
anomaly_mask = np.zeros(N_TOTAL, dtype=bool)

# Inject anomalies into the TEST segment only
# 1. Spike burst at t=5.2–5.25 s (bearing fault)
spike_start = N_TRAIN + 200
spike_end   = N_TRAIN + 250
signal[spike_start:spike_end] += rng.uniform(2.0, 4.0, size=spike_end - spike_start).astype(np.float32)
anomaly_mask[spike_start:spike_end] = True

# 2. Slow thermal drift at t=5.5–6.0 s
drift_start = N_TRAIN + 500
drift_end   = N_TRAIN + 1000
drift = np.linspace(0, 1.8, drift_end - drift_start).astype(np.float32)
signal[drift_start:drift_end] += drift
anomaly_mask[drift_start:drift_end] = True

# 3. Sensor clamp-out at t=6.5–6.6 s (flat-line)
clamp_start = N_TRAIN + 1500
clamp_end   = N_TRAIN + 1600
signal[clamp_start:clamp_end] = 0.0
anomaly_mask[clamp_start:clamp_end] = True

print(f'Total samples : {N_TOTAL} ({TRAIN_SEC}s train + {TEST_SEC}s test)')
print(f'Anomaly frames: {anomaly_mask.sum()} ({100*anomaly_mask.mean():.1f}%)')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(t, normal, color='steelblue', linewidth=0.5, label='Normal baseline')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Normal Signal Baseline (50/100/150 Hz)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, signal, color='steelblue', linewidth=0.5, label='Signal with anomalies')
# Shade anomalous regions
in_anomaly = False
start = 0
for i in range(N_TOTAL):
    if anomaly_mask[i] and not in_anomaly:
        start = i; in_anomaly = True
    elif not anomaly_mask[i] and in_anomaly:
        axes[1].axvspan(t[start], t[i], color='coral', alpha=0.4)
        in_anomaly = False
if in_anomaly:
    axes[1].axvspan(t[start], t[-1], color='coral', alpha=0.4)
axes[1].axvline(TRAIN_SEC, color='gray', linewidth=1.5, linestyle='--', label='Train/test boundary')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Signal with Injected Anomalies (shaded = ground truth)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/kizzasi_anomaly_signal.png', dpi=120)
plt.show()

## 2. Configure and Warm Up the Predictor

We use the `sensor` preset (single-sensor variant) and warm up the hidden state on the training portion of the signal. Warming up is equivalent to "learning" the normal pattern — no gradient descent required for SSM inference-time adaptation.

In [ ]:
WARMUP_FRAC = 0.8   # use 80% of training segment for warmup
WARMUP_STEPS = int(N_TRAIN * WARMUP_FRAC)
CALIB_STEPS  = N_TRAIN - WARMUP_STEPS  # remaining 20% for threshold calibration

if KIZZASI_AVAILABLE:
    # sensor(num_sensors=1) = 1-in, 1-out predictor optimised for sensor streams
    config = kizzasi.Config.sensor(num_sensors=1)
    predictor = kizzasi.Predictor(config)
    print(predictor)

    # --- Warmup phase ---
    for i in range(WARMUP_STEPS):
        x = np.array([signal[i]], dtype=np.float32)
        predictor.step(x)

    print(f'Warmup complete: {WARMUP_STEPS} steps')

    # --- Calibration phase: estimate normal error distribution ---
    calib_errors = []
    for i in range(WARMUP_STEPS, N_TRAIN):
        x = np.array([signal[i]], dtype=np.float32)
        pred = predictor.step(x)
        true_next = signal[i + 1] if i + 1 < len(signal) else signal[i]
        calib_errors.append(abs(true_next - float(pred[0])))

    calib_errors = np.array(calib_errors)
    THRESHOLD = float(np.mean(calib_errors) + 3.0 * np.std(calib_errors))
    print(f'Calibration MAE : {calib_errors.mean():.5f}')
    print(f'Anomaly threshold (mu + 3 sigma): {THRESHOLD:.5f}')
else:
    # Stub: derive threshold from analytic knowledge of the normal signal
    THRESHOLD = 0.25
    print(f'(stub) Anomaly threshold: {THRESHOLD:.5f}')

## 3. Attach Constraint Guardrails

We add a physical hard constraint: vibration amplitude outside `[-3.0, 3.0]` is physically impossible for this sensor. Any prediction or observation violating this range is immediately flagged, regardless of the learned threshold.

In [ ]:
PHYSICAL_MIN = -3.0
PHYSICAL_MAX =  3.0

if KIZZASI_AVAILABLE:
    amplitude_guard = kizzasi.ConstraintSpec(
        name='vibration_amplitude',
        min_val=PHYSICAL_MIN,
        max_val=PHYSICAL_MAX,
        hard_reject=False,   # soft: clip predictions into range
    )
    predictor.set_guardrails([amplitude_guard])
    print(f'Guardrails active: {predictor.has_guardrails()}')
    print(f'Physical constraint: [{PHYSICAL_MIN}, {PHYSICAL_MAX}]')
else:
    print(f'(stub) ConstraintSpec vibration_amplitude [{PHYSICAL_MIN}, {PHYSICAL_MAX}]')

## 4. Run Anomaly Detection on the Test Segment

For each test sample we:
1. Feed it to `predictor.step()` — the guardrail clips the *prediction* if it violates physical bounds
2. Compute `|predicted_next - actual_next|`
3. Flag as anomalous if error > threshold OR if the *observation* itself violates physical bounds

In [ ]:
test_errors    = np.zeros(N_TEST, dtype=np.float32)
test_predicted = np.zeros(N_TEST, dtype=np.float32)
constraint_flags = np.zeros(N_TEST, dtype=bool)  # immediate physical violation

if KIZZASI_AVAILABLE:
    for i in range(N_TEST):
        idx = N_TRAIN + i
        obs = signal[idx]

        # Constraint check on observed value (not just predicted)
        constraint_flags[i] = (obs < PHYSICAL_MIN) or (obs > PHYSICAL_MAX)

        x = np.array([obs], dtype=np.float32)
        pred = predictor.step(x)          # guardrail clips the output
        pred_val = float(pred[0])
        test_predicted[i] = pred_val

        true_next = signal[idx + 1] if idx + 1 < len(signal) else obs
        test_errors[i] = abs(true_next - pred_val)
else:
    # Stub: use analytic sine wave + amplified errors at injection points
    t_test = t[N_TRAIN:]
    analytic_pred = (
        0.6 * np.sin(2 * np.pi * 50  * t_test) +
        0.3 * np.sin(2 * np.pi * 100 * t_test)
    ).astype(np.float32)
    test_predicted = analytic_pred
    test_errors    = np.abs(signal[N_TRAIN:] - analytic_pred)
    # Amplify error at true anomaly positions to simulate detection
    local_mask = anomaly_mask[N_TRAIN:]
    test_errors[local_mask] *= rng.uniform(3.0, 8.0, size=local_mask.sum()).astype(np.float32)
    constraint_flags = signal[N_TRAIN:] > PHYSICAL_MAX

# --- Combined detection mask ---
error_flags = test_errors > THRESHOLD
detected    = error_flags | constraint_flags

# Evaluate against ground truth
gt_test = anomaly_mask[N_TRAIN:]
tp = int((detected & gt_test).sum())
fp = int((detected & ~gt_test).sum())
fn = int((~detected & gt_test).sum())
precision = tp / (tp + fp + 1e-9)
recall    = tp / (tp + fn + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)

print(f'Threshold           : {THRESHOLD:.5f}')
print(f'Error flags (stat.) : {error_flags.sum()}')
print(f'Constraint flags    : {constraint_flags.sum()}')
print(f'Combined detections : {detected.sum()}')
print(f'Ground truth anoms  : {gt_test.sum()}')
print(f'Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}')

## 5. Visualise Results

In [ ]:
t_test = t[N_TRAIN:]
test_sig = signal[N_TRAIN:]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# ---- Panel 1: raw signal with detection overlay ----
axes[0].plot(t_test, test_sig, color='steelblue', linewidth=0.7, label='Observed signal')
axes[0].plot(t_test, test_predicted, color='orange', linewidth=0.7,
             alpha=0.8, linestyle='--', label='Kizzasi prediction')

# Shade ground-truth anomaly windows
in_anom = False
astart  = 0
for i in range(N_TEST):
    if gt_test[i] and not in_anom:
        astart = i; in_anom = True
    elif not gt_test[i] and in_anom:
        axes[0].axvspan(t_test[astart], t_test[i], color='coral', alpha=0.3)
        in_anom = False
if in_anom:
    axes[0].axvspan(t_test[astart], t_test[-1], color='coral', alpha=0.3)

# Mark detections
det_idx = np.where(detected)[0]
if len(det_idx):
    axes[0].scatter(t_test[det_idx], test_sig[det_idx],
                    s=8, color='red', zorder=5, label='Detected', alpha=0.6)

axes[0].axhline(PHYSICAL_MAX, color='purple', linewidth=1, linestyle=':', label='Physical bound')
axes[0].axhline(PHYSICAL_MIN, color='purple', linewidth=1, linestyle=':')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Anomaly Detection — Vibration Sensor')
axes[0].legend(fontsize=8, loc='upper right')
axes[0].grid(True, alpha=0.3)

# ---- Panel 2: prediction error + threshold ----
axes[1].plot(t_test, test_errors, color='mediumseagreen', linewidth=0.7)
axes[1].axhline(THRESHOLD, color='firebrick', linewidth=1.5, linestyle='--',
                label=f'Threshold = {THRESHOLD:.4f}')
axes[1].set_ylabel('|Error|')
axes[1].set_title('Prediction Error (statistical detection)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# ---- Panel 3: detection bitmask (TP / FP / FN / TN) ----
color_map = np.zeros((N_TEST, 3))
# TN = white, TP = green, FP = orange, FN = red
tp_mask = detected & gt_test
fp_mask = detected & ~gt_test
fn_mask = ~detected & gt_test

for i in range(N_TEST):
    if tp_mask[i]:   color_map[i] = [0.2, 0.8, 0.2]   # green
    elif fp_mask[i]: color_map[i] = [1.0, 0.6, 0.0]   # orange
    elif fn_mask[i]: color_map[i] = [0.9, 0.1, 0.1]   # red
    else:            color_map[i] = [0.9, 0.9, 0.9]   # light gray

axes[2].imshow(color_map[np.newaxis], aspect='auto',
               extent=[t_test[0], t_test[-1], 0, 1])
legend_patches = [
    mpatches.Patch(color=(0.2, 0.8, 0.2), label=f'TP={tp}'),
    mpatches.Patch(color=(1.0, 0.6, 0.0), label=f'FP={fp}'),
    mpatches.Patch(color=(0.9, 0.1, 0.1), label=f'FN={fn}'),
    mpatches.Patch(color=(0.9, 0.9, 0.9), label='TN'),
]
axes[2].legend(handles=legend_patches, fontsize=8, loc='lower right')
axes[2].set_xlabel('Time (s)')
axes[2].set_title(f'Detection Map  (P={precision:.3f}  R={recall:.3f}  F1={f1:.3f})')
axes[2].set_yticks([])

plt.tight_layout()
plt.savefig('/tmp/kizzasi_anomaly_detection.png', dpi=120)
plt.show()
print('Results saved to /tmp/kizzasi_anomaly_detection.png')

## 6. Calibrating the Threshold — Precision-Recall Curve

In [ ]:
thresholds = np.linspace(0.001, test_errors.max() * 1.1, 200)
precisions = []
recalls    = []
f1s        = []

for thr in thresholds:
    det = (test_errors > thr) | constraint_flags
    tp_ = int((det & gt_test).sum())
    fp_ = int((det & ~gt_test).sum())
    fn_ = int((~det & gt_test).sum())
    p   = tp_ / (tp_ + fp_ + 1e-9)
    r   = tp_ / (tp_ + fn_ + 1e-9)
    f1_ = 2 * p * r / (p + r + 1e-9)
    precisions.append(p)
    recalls.append(r)
    f1s.append(f1_)

best_idx = int(np.argmax(f1s))
best_thr = float(thresholds[best_idx])
best_f1  = float(f1s[best_idx])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(recalls, precisions, color='steelblue', linewidth=1.5)
axes[0].scatter([recalls[best_idx]], [precisions[best_idx]],
                color='red', zorder=5, s=60, label=f'Best F1={best_f1:.3f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, f1s, color='mediumseagreen', linewidth=1.5, label='F1')
axes[1].plot(thresholds, precisions, color='steelblue',    linewidth=1, linestyle='--', label='Precision')
axes[1].plot(thresholds, recalls,    color='coral',        linewidth=1, linestyle='--', label='Recall')
axes[1].axvline(best_thr, color='red', linewidth=1.2, linestyle=':',
                label=f'Best threshold={best_thr:.4f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('F1 / Precision / Recall vs Threshold')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/kizzasi_pr_curve.png', dpi=120)
plt.show()

print(f'Best threshold : {best_thr:.5f}')
print(f'Best F1        : {best_f1:.3f}')

## 7. Multi-Sensor Extension

Kizzasi's `sensor` preset natively supports multiple channels. Here we sketch the API for a 3-axis accelerometer.

In [ ]:
N_SENSORS = 3  # x, y, z axes

if KIZZASI_AVAILABLE:
    multi_cfg = kizzasi.Config.sensor(num_sensors=N_SENSORS)
    multi_pred = kizzasi.Predictor(multi_cfg)
    print(multi_pred)

    # Per-axis guardrails
    guards = [
        kizzasi.ConstraintSpec('accel_x', min_val=-5.0, max_val=5.0, dimension=0),
        kizzasi.ConstraintSpec('accel_y', min_val=-5.0, max_val=5.0, dimension=1),
        kizzasi.ConstraintSpec('accel_z', min_val=-9.8, max_val=9.8, dimension=2),  # gravity
    ]
    multi_pred.set_guardrails(guards)
    print(f'Multi-sensor guardrails active: {multi_pred.has_guardrails()}')

    # Warm up and step with 3-D input
    for _ in range(200):
        x3 = rng.standard_normal(N_SENSORS).astype(np.float32) * 0.5
        multi_pred.step(x3)

    pred3 = multi_pred.step(np.zeros(N_SENSORS, dtype=np.float32))
    print(f'3-sensor prediction shape: {pred3.shape}  values: {pred3}')
else:
    print(f'(stub) 3-sensor Config.sensor({N_SENSORS})')
    print('(stub) Per-axis ConstraintSpec with dimension=0/1/2')
    print('(stub) predictor.step(np.zeros(3)) -> shape (3,)')

## Summary

### Detection Strategy

```
for each new sample obs:
    pred = predictor.step(obs)           # O(1) SSM update
    error = |next_obs - pred|
    if error > threshold:                # statistical anomaly
        alert("prediction error exceeded")
    if obs outside physical_bounds:      # constraint anomaly
        alert("physical constraint violated")
```

### Key Hyperparameters

| Parameter | Role |
|-----------|------|
| `hidden_dim` | SSM capacity — larger = captures more complex periodicity |
| `num_layers` | Depth — 2–4 usually sufficient for sensor signals |
| `warmup_steps` | How many samples to feed before calibrating threshold |
| `threshold = mu + k*sigma` | `k=3` gives ~0.3% false-positive rate for Gaussian errors |
| `ConstraintSpec(hard_reject=False)` | Soft: clip predictions into range |
| `ConstraintSpec(hard_reject=True)` | Hard: raise `RuntimeError` on violation |

### Next Steps

- Load a pre-trained Kizzasi checkpoint: `predictor = kizzasi.Predictor.from_checkpoint('model.safetensors')` _(planned v0.3)_
- Stream directly from MQTT: `kizzasi_io::MqttClient` (Rust API, Python bridge in progress)
- Export anomaly events to Prometheus / InfluxDB for dashboarding